# Import Everything 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import xarray as xr
import math
from tkinter.filedialog import askopenfilename, askopenfilenames
from lmfit.models import GaussianModel
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
import lmfit
from scipy.stats import norm
from scipy.interpolate import interp1d
%matplotlib widget

# allow multiple outputs in one cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

def poly2(x, a, b, c):
    return a * x**2 + b * x + c


def poly4(x, a, b, c, d, e):
    return a * x**4 + b * x**3 + c * x**2 + d * x + e

def poly3(x, a, b, c, d):
    return a * x**3 + b * x**2 + c * x + d

def gaussian(x, a, x0, sigma):
    return a * np.exp(-(x - x0)**2 / (2 * sigma**2))

def gaussian_fwhm(sigma):
    fwhm = 2 * math.sqrt(2 * math.log(2)) * sigma
    return fwhm


def heatmap_interactive(_x, _y, _data, _title, _cmap='jet', _symlog=False):
    # Create a figure with specified size
    fig = plt.figure(figsize=(8, 8))

    # Define a grid layout for the subplots
    gs = gridspec.GridSpec(2, 2, width_ratios=[1, 0.5], height_ratios=[0.5, 1], hspace=0.2, wspace=0.2)

    # Create the main plot in the bottom-left corner of the grid
    ax_main = plt.subplot(gs[1, 0])
    main_plot = ax_main.pcolormesh(_x, _y, _data, cmap=_cmap)
    ax_main.set(xlabel='Delay / ps', ylabel='Wavelength / nm')
    # Set mixed log-lin scale with threshold value linthresh if symlog is True
    if _symlog:
        ax_main.set_xscale('symlog', linthresh=0.01)
    # Set axis range to min and max values
    ax_main.set_xlim(_x[0],_x[-1])
    ax_main.set_ylim(_y[0],_y[-1])

    # Create the kinetic plot in the top-left corner of the grid
    ax_kin = plt.subplot(gs[0, 0])
    line_kin, = ax_kin.plot(_x,np.zeros(_x.shape))
    kin_zero_line, = ax_kin.plot([_x[0],_x[-1]],[0,0], color="0.6")
    ax_kin.set_xlim(_x[0],_x[-1])
    if _symlog:
        ax_kin.set_xscale('symlog', linthresh=0.01)

    # Create the spectrum plot in the bottom-right corner of the grid
    ax_spec = plt.subplot(gs[1, 1])
    line_spec, = ax_spec.plot(np.zeros(_y.shape),_y)
    spec_zero_line, = ax_spec.plot([0,0],[_y[0],_y[-1]], color="0.6")
    ax_spec.set_ylim(_y[0],_y[-1])

    # This lower bounds list is necessary because the blocks in the 2D-plot cover a certain range
    def create_lower_bounds(_value_list):
        result = np.empty_like(_value_list)
        # First lower bound is equal to the lowest value in the nm-list
        result[0] = _value_list[0]
        # Example: lower bound for 100 ps is 97.5 ps if the value prior is 95 ps, and 75 ps if the value prior is 50 ps.
        for i in range(1,len(_value_list)):
            result[i] = (_value_list[i]+_value_list[i-1])/2
        return result

    nm_lower_bounds = create_lower_bounds(_y)
    time_lower_bounds = create_lower_bounds(_x)

    def nm_to_index(_nm):
        return np.where(_nm > nm_lower_bounds)[0][-1]

    def time_to_index(_time):
        return np.where(_time > time_lower_bounds)[0][-1]

    def mouse_move(event):
        x = event.xdata
        y = event.ydata
        if x is not None and y is not None:
            if x>=_x[0] and x<=_x[-1] and y>=_y[0] and y<=_y[-1]:

                # Update spectra slice and rescale
                new_spec = _data[:,time_to_index(x)]
                line_spec.set_xdata(new_spec)
                spec_bounds = ax_spec.get_ylim()
                spec_range = new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].max()-new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].min()
                ax_spec.set_xlim(new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].min()-0.1*spec_range,new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].max()+0.1*spec_range)

                # Update kinetic slice and rescale
                new_kin = _data[nm_to_index(y),:]
                line_kin.set_ydata(new_kin)
                kin_bounds = ax_kin.get_xlim()
                kin_range = new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].max()-new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].min()
                ax_kin.set_ylim(new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].min()-0.1*kin_range,new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].max()+0.1*kin_range)

                # Redraw figure
                fig.canvas.draw_idle()

    # Connect mouse move event to the figure
    fig.canvas.mpl_connect('motion_notify_event', mouse_move)

    # Find max absolute value of 2D data in the specified zoom mode of the plot
    def get_maxvalue(_xlim, _ylim, _xvals, _yvals, _data_array):
        y_filter = (_yvals>=_ylim[0]) & (_yvals<=_ylim[1])
        x_filter = (_xvals>=_xlim[0]) & (_xvals<=_xlim[1])

        if not np.all(y_filter == False) and not np.all(x_filter == False):
            return np.amax(np.abs(_data_array[y_filter][:,x_filter]))
        else:
            return 0

    def on_xlims_change(event_ax):
        ax_kin.set_xlim(event_ax.get_xlim())

        new_max = get_maxvalue(event_ax.get_xlim(),event_ax.get_ylim(),_x,_y,_data)
        if new_max > 0:
            main_plot.set_clim(vmin=-new_max, vmax=new_max)

    def on_ylims_change(event_ax):
        ax_spec.set_ylim(event_ax.get_ylim())

        new_max = get_maxvalue(event_ax.get_xlim(),event_ax.get_ylim(),_x,_y,_data)
        if new_max > 0:
            main_plot.set_clim(vmin=-new_max, vmax=new_max)

    # Connect axis limits change events to the respective functions
    ax_main.callbacks.connect('xlim_changed', on_xlims_change)
    ax_main.callbacks.connect('ylim_changed', on_ylims_change)

    # Show the plot
    plt.show(block=False)


def lin(x,a,b):
    return a*x+b


def smooth_data_average_window(data, window_size):
    # Calculate rolling mean with specified window size
    windows = data.rolling(spectral= window_size ,center = True, min_periods=1)
    moving_average = windows.mean()
    return moving_average

def calculate_fwhm(spectral_data, time_coords):
    #spectral_data = np.abs(spectral_data)
    peak_value = np.max(spectral_data)
    half_max = peak_value / 2.0
    indices_above_half = np.where(spectral_data >= half_max)[0]
    
    if len(indices_above_half) < 2:
        return np.nan  # Not enough data to calculate FWHM
    
    max_value = np.where(spectral_data == peak_value )[0]
    valid_indices = [max_value[0]] #Create an where where we gonna remove the data that are above the half but that don't correspond to the peak
    current_value = max_value[0]
    for i in  np.where(indices_above_half > max_value)[0]: #Look in the positive increment  
        if indices_above_half[i] == current_value + 1:
            valid_indices.append(indices_above_half[i])
            current_value += 1
        else:
            break
    current_value = max_value[0]
    for i in  reversed(np.where(indices_above_half < max_value)[0]): #Look in the negative increment  
        if indices_above_half[i] == current_value - 1:
            valid_indices.insert(0,indices_above_half[i])
            current_value -= 1
        else:
            break

    # Interpolate to get a more accurate crossing point
    f_left = interp1d(spectral_data[valid_indices[0]-1:valid_indices[0]+1], time_coords[valid_indices[0]-1:valid_indices[0]+1], kind='linear')
    f_right = interp1d(spectral_data[valid_indices[-1]:valid_indices[-1]+2], time_coords[valid_indices[-1]:valid_indices[-1]+2], kind='linear')
    left_half_max_time = f_left(half_max)
    right_half_max_time = f_right(half_max)
    
    # Calculate FWHM as the difference between these two time points
    fwhm = right_half_max_time - left_half_max_time

    return fwhm 


# Import Previouslt saved Xarray

In [ ]:
####################################################################################### 
################ Import a previous fit of the Coherent artifact #######################

path_to_xarray = '/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu/Data/Replicate/Dataset_fit_C.nc'
dataset_fit_CA = xr.open_dataset(path_to_xarray)




# Calibration

In [ ]:
#Wavelength Calibaration
pixels = np.arange(0, 2048)
pixels_calibation = [583,685,1274,1490]
wv_calibation = [447,469,605,655]

calibration,_ = curve_fit(lin,pixels_calibation,wv_calibation, method= "dogbox")
Fit_calibration = lin(pixels,*calibration)

plt.figure()
plt.plot(pixels_calibation,wv_calibation,  'o', label='Data')
plt.plot(pixels,Fit_calibration, '-', label='Fit')
plt.xlabel('Pixels)')
plt.ylabel('Wavelength (nm)')
plt.legend()
plt.grid("On")
plt.show()
print(f"y = {calibration[0]}x + {calibration[1]}")

lambda_values = calibration[0] * pixels + calibration[1]
#lambda_values = pixels

# Step 1: Import the Solvent scan

In [ ]:
time_file = askopenfilename(filetypes=[("Text files", "*.txt")], title="Select Time vector data")
time_solvent = np.loadtxt(time_file)
time_solvent = time_solvent / 1000  # convert from fs to ps

# Get TA scan files
ta_scan_files = askopenfilenames(filetypes=[("Text files", "*.txt")], title="Select solvent scan file(s)")

if len(ta_scan_files) > 1:
    Full_Data = np.zeros((2048, len(time_solvent), len(ta_scan_files))) #Array that will contain all the 
    for n, file in enumerate(ta_scan_files):
        data = np.loadtxt(file)
        Full_Data[:, :, n] = data  # save data array to 3D array (lambda, time, scan)
    
    scan_solvent = np.mean(Full_Data, axis=2)
    scan_solvent = scan_solvent - np.mean(scan_solvent[:,1:5],axis=1)[:, np.newaxis] #Baseline correction 
else:
    scan_solvent = np.loadtxt(ta_scan_files[0])
# Now 'scan' contains the processed data

#Create an xarray to manipulate the data (much easier)
dataset_solvent = xr.Dataset(
    {
        "data": (["time","spectral",], np.transpose(scan_solvent))
    },
    coords={
        "time": time_solvent,
        "spectral": lambda_values
    }
)

# Print the dataset
print(dataset_solvent)

heatmap_interactive(dataset_solvent.time, dataset_solvent.spectral, dataset_solvent['data'].transpose('spectral','time'),'Averaged scan plot',_symlog=False)

In [ ]:
heatmap_interactive(dataset_solvent.time, dataset_solvent.spectral, dataset_solvent['data'].transpose('spectral','time'),'Averaged scan plot',_symlog=False)

# Step 2 - Get IRF information from it  

In [ ]:
lower_signal_bound = 430
higher_singal_bound = 565
threshold = 0.0035

wl_range = np.arange(lower_signal_bound,higher_singal_bound+1,1)



ROI = smooth_data_average_window(dataset_solvent.sel(spectral=slice(lower_signal_bound, higher_singal_bound)),35)

ROI_FWHM_values = []
gaussian_fit_param = np.zeros([np.shape(wl_range)[0],3])
i = 0


for spectral_value in wl_range: 
    # Extract the kinetic trace for the selected spectral value
    kinetic_trace = np.abs(ROI['data'].sel(spectral=spectral_value, method='nearest'))

    # Find peaks in the kinetic trace
    peaks, _ = find_peaks(kinetic_trace)
    filtered_peaks = peaks[kinetic_trace[peaks] > threshold]

    # Ensure we have at least two peaks
    if len(filtered_peaks) >= 2:
        # Identify the two outermost peaks
        first_peak = filtered_peaks[0]
        last_peak = filtered_peaks[-1]
    # Create a mask where values between the two outermost peaks are removed
        mask = np.ones_like(kinetic_trace, dtype=bool)  
        mask[first_peak:last_peak+1] = False  
        
        # Apply the mask to the data
        sliced_kinetics = kinetic_trace.where(mask) 

        # Fit a Gaussian to the sliced data
        time_data = time_data = np.concatenate([sliced_kinetics.time[0:first_peak].values, sliced_kinetics.time[last_peak+1:].values], axis=0)
        kinetic_data = sliced_kinetics.data[~np.isnan(sliced_kinetics.data)] #Remove the NaN values, needed for the fit 

        # Define Gaussian model
        gauss_model = GaussianModel()

        # Set initial parameters
        gap_center = (sliced_kinetics.time[first_peak].values + sliced_kinetics.time[last_peak].values) / 2
        params = gauss_model.make_params(amplitude=ROI['data'].max().item(), center=gap_center, sigma=np.std(time_data))

        # Fit the model to the data
        fit_result = gauss_model.fit(kinetic_data, params, x=time_data) 

        # Generate smooth fitted curve
        smooth_time = np.linspace(time_data.min(), time_data.max(), 300)
        fitted_curve = gauss_model.eval(fit_result.params, x=smooth_time)

        #Print fitted parameters
        fitted_params = fit_result.params
        gaussian_fit_param[i][0] = fitted_params['amplitude'].value
        gaussian_fit_param[i][1] = fitted_params['center'].value
        gaussian_fit_param[i][2] = fitted_params['sigma'].value
        ROI_FWHM_values.append(gaussian_fwhm(fitted_params['sigma'].value*1000))
        i = i+1
    else:
        fwhm = calculate_fwhm(kinetic_trace, kinetic_trace.time.values)*1000 
        ROI_FWHM_values.append(fwhm)
        i = i+1

fwhm_trend,_ = curve_fit(lin,wl_range,ROI_FWHM_values, method= "dogbox")
fwhm_lin = lin(wl_range,*fwhm_trend)
plt.figure(figsize=(12, 6))
plt.scatter(wl_range,ROI_FWHM_values,color='blue', marker='x')
plt.plot(wl_range,fwhm_lin, '--', label='Linear Fit') 
plt.legend()
plt.ylim([0,400])
plt.title('IRF estimation with a Gaussian Fit')
plt.grid('On')
plt.xlabel("Wavelength (nm)")
plt.ylabel("FWHM (fs)")




In [ ]:
def poly2(x, a, b, c):
    return a * x**2 + b * x + c


def poly4(x, a, b, c, d, e):
    return a * x**4 + b * x**3 + c * x**2 + d * x + e

def poly3(x, a, b, c, d):
    return a * x**3 + b * x**2 + c * x + d

params_position_IRF, _ = curve_fit(poly4, wl_range, gaussian_fit_param[:,1])
Fit_coherence = poly4(wl_range,*params_position_IRF)


plt.figure(figsize=(12, 6))
plt.scatter(gaussian_fit_param[:,1],wl_range,color='blue', marker='x')
plt.plot(Fit_coherence,wl_range,  '-', label='Fit', color = 'red')
plt.title('IRF estimation with a Gaussian Fit')
plt.grid('On')
plt.ylabel("Wavelength (nm)")
plt.xlabel("position CA)")

# Step 3 - Get CA paramters 

### Single position - Use it to manually determine goood parameters for the lowest region of your fit (eg . 469)

In [ ]:
def Fcos(params, time): 
    B = params['B']
    Phi = params['Phi']
    A0 = params['A0']
    A1 = params['A1']
    tau = params['CA_FWHM']
    dt0 = params['CA_center']
    
    cos_term = np.cos(B * (time - dt0)**2 + Phi) 
    exp_term = A0 * np.exp(-4 * np.log(2) * ((time - dt0) ** 2) / (tau ** 2))
    correction_term = A1 * (8 * np.log(2) / (tau ** 2)) * (time - dt0) * np.exp(-4 * np.log(2) * ((time - dt0) ** 2) / (tau ** 2))
    
    result = cos_term * (exp_term - correction_term)
    
    return result

def Gaussian(params,t):
    c = params['c']
    sigma = params['sigma']
    gauss = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-((t - c) ** 2) / (2 * sigma ** 2))
    gauss_first_derivative = ((c - t) / (sigma ** 3 * np.sqrt(2 * np.pi))) * np.exp(-((t - c) ** 2) / (2 * sigma ** 2))
    gauss_second_derivative = ((t ** 2 - 2 * c * t - sigma ** 2 + c ** 2) / (sigma ** 5 * np.sqrt(2 * np.pi))) * np.exp(-((t - c) ** 2) / (2 * sigma ** 2))
    return gauss + gauss_first_derivative + gauss_second_derivative



def residuals_Fcos(params, time, data):
    model = Fcos(params, time)
    return np.sqrt((model - data)**2)

def residuals_gauss(params, t, data):
    model = Gaussian(params, t)
    return model - data


In [ ]:
wv_1 = 469 #in nm

xml_region = dataset_solvent.data.sel(spectral=slice(430, 560)).sel(time=slice(-1,2))
single_trace = xml_region.sel(spectral = [wv_1], method="nearest")
max_time = single_trace['time'][single_trace.argmax().item()].item()
single_trace = single_trace.sel(time = slice(max_time-0.5,max_time+0.5)) #Looks at -500 and +500 fs around the CA peak 
time = single_trace.time.values
data = single_trace.values.reshape(np.shape(single_trace.data)[0])

# Set up lmfit parameters
params = lmfit.Parameters()
params.add('B', value=  200.246245 , min = 0,vary = True) #determines how quickly the fringe periodicity increases 
params.add('Phi', value=155.508536   ,vary = True) #phase shift of the modulation 
params.add('A0', value=  -0.06680426     , vary = True) #Amplitude main Gaussian. Positve results in main peak down 
params.add('A1', value=5.3965e-04, vary = True) # help fit of asymmetric artifacts 
params.add('CA_FWHM', value= lin(469,*fwhm_trend)/1000,min=0, vary = True) #FWHM 
params.add('CA_center', value=  poly4(469,*params_position_IRF), vary = True) #Center of the chirp 


minimizer = lmfit.Minimizer(residuals_Fcos, params, fcn_args=(time, data))
result_init = minimizer.minimize()

x_lims = [max_time-0.2,max_time+0.3]
fig, (ax, ax2) = plt.subplots(nrows=2, gridspec_kw={'height_ratios': [3, 1]}, figsize=(12, 8))
ax.scatter(time, data, color='grey', s=25, label='Coherent Artifact', alpha=0.5)
ax.plot(time, Fcos(result_init.params, time), color='blue', linewidth=3, label='Modeling of the Artifact')
ax.set_xlim(x_lims)


residuals = data - Fcos(result_init.params, time)
ax2.plot(time, residuals, color='grey', label='Residual', alpha=0.5)

ax.set_xlabel("Time (ps)",fontsize=20)
ax.set_ylabel("ΔA",fontsize=20)
ax2.set_xlabel("Time (ps)",fontsize=20)
ax2.set_ylabel("Residual",fontsize=20)  
ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax2.set_xlim(ax.get_xlim()) 
ax2.set_ylim(ax.get_ylim()) 
ax.tick_params(axis='both', which='major', labelsize=15)
ax2.tick_params(axis='both', which='major', labelsize=15)

handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax.legend(handles,labels, loc='upper right',fontsize = 20)
ax2.legend(handles2, labels2, loc='upper right',fontsize = 20)
fig.suptitle(f"Modeling Coherent Artifact, solvent scan, {wv_1} nm",fontsize=25)
plt.tight_layout()



# Print the fit report
print(lmfit.fit_report(result_init))

# Use the previouly determine parameters to Fit the full region

In [ ]:
lower_end = wv_1
higher_end = 560

xml_region = dataset_solvent.data.sel(spectral=slice(wv_1, 560)).sel(time=slice(-1,2))

fit_CA_param = {}
fit_CA_result = np.transpose(np.zeros_like(xml_region))

for i, wavelength in enumerate(xml_region.spectral):
    single_trace = xml_region.sel(spectral=[wavelength], method="nearest")
    time = single_trace.time.values
    data = single_trace.values.reshape(np.shape(single_trace.data)[0])

    if i == 0:
        # Set up lmfit parameters for the first wavelength

        params = lmfit.Parameters()
        params.add('B', value= result_init.params['B'].value , min = 0,vary = False) #determines how quickly the fringe periodicity increases 
        params.add('Phi', value=result_init.params['Phi'].value  ,vary = False) #phase shift of the modulation 
        params.add('A0', value=  result_init.params['A0'].value     , vary = False) #Amplitude main Gaussian. Positve results in main peak down 
        params.add('A1', value=result_init.params['A1'].value, vary = False) # help fit of asymmetric artifacts 
        params.add('CA_FWHM', value= result_init.params['CA_FWHM'].value,min=0, vary = False) #FWHM 
        params.add('CA_center', value= result_init.params['CA_center'].value, vary = False) #Center of the chirp 
    else:
        # Use the previous wavelength's fitted parameters as initial values
        prev_wavelength = xml_region.spectral[i - 1].item()
        prev_params = fit_CA_param[prev_wavelength]
        params = lmfit.Parameters()
        for name, param in prev_params.items():
            params.add(name, value=param.value, vary=True) #, min=param.value*0.5, max=param.value*1.5

    # Perform minimization
    minimizer = lmfit.Minimizer(residuals_Fcos, params, fcn_args=(time, data))
    result = minimizer.minimize()

    # Store the fit result in the dictionary to be used for the next wavelength
    fit_CA_param[wavelength.item()] = result.params
    fit_CA_result[i] = Fcos(result.params, time)


In [ ]:
dataset_fit_CA = xr.Dataset(
    {
        "data": (["time","spectral",], np.transpose(fit_CA_result))
    },
    coords={
        "time": xml_region.time,
        "spectral": xml_region.spectral
    }
)

# Plot visualization

In [ ]:
xml_region = dataset_solvent.data.sel(spectral=slice(lower_end, higher_end)).sel(time=slice(-1,2))
# Assuming dataset_fit and dataset_fit2 are already defined
fig, axes = plt.subplots(2, 2, figsize=(12, 8))  # 1 row, 2 columns

# Plot the first dataset
dataset_fit_CA['data'].plot.imshow(ax=axes[0, 0])
axes[0, 0].set_title("Fit",fontsize=25)
axes[0, 0].set_aspect("auto")  # Set aspect ratio
axes[0, 0].set_xlabel("Time (ps)",fontsize=20)
axes[0, 0].set_ylabel("∆A",fontsize=20)
axes[0, 0].tick_params(axis='both', which='major', labelsize=20)


# Plot the second dataset
xml_region.plot.imshow(ax=axes[0, 1])
axes[0, 1].set_title("Solvent Scan",fontsize=25)
axes[0, 1].set_aspect("auto")  # Set aspect ratio
axes[0, 1].set_xlabel("Time (ps)",fontsize=20)
axes[0, 1].set_ylabel("∆A",fontsize=20)
axes[0, 1].tick_params(axis='both', which='major', labelsize=20)

diff = xml_region - dataset_fit_CA
diff['data'].plot.imshow(ax=axes[1, 0])
axes[1, 0].set_title("Difference",fontsize=25)
axes[1, 0].set_aspect("auto")  # Set aspect ratio
axes[1, 0].set_xlabel("Time (ps)",fontsize=20)
axes[1, 0].set_ylabel("∆A",fontsize=20)
axes[1, 0].tick_params(axis='both', which='major', labelsize=20)

# Hide the second subplot in the second row
axes[1, 1].axis('off')



plt.tight_layout()
plt.show();

Q1 = np.sqrt(np.mean((xml_region.data - dataset_fit_CA.data)**2)) / np.sqrt(np.mean((xml_region.data)**2))
print(Q1.values)

In [ ]:
fit_CA_param_scan1 

In [ ]:
import pandas as pd

# Suppose your dictionary is named `fit_results`
data = {}

for wavelength, params in fit_CA_param_scan2.items():
    data[wavelength] = {name: par.value for name, par in params.items()}

# Convert to DataFrame and transpose
df = pd.DataFrame(data).T  # .T makes wavelengths the index, parameters the columns
df.index.name = 'Wavelength'

# Save to Excel
df.to_excel("fit_results_scan2.xlsx")


### Visualize sigle kinetics of the plot 

In [ ]:
wv = 557 #in nm

xml_region = dataset_solvent.data.sel(spectral=slice(lower_end, higher_end)).sel(time=slice(-1,2))
single_trace = xml_region.sel(spectral = [wv], method="nearest")
max_time = single_trace['time'][single_trace.argmax().item()].item()
single_trace = single_trace.sel(time = slice(max_time-0.5,max_time+0.5)) #Looks at -500 and +500 fs around the CA peak 
time = single_trace.time.values
data = single_trace.values.reshape(np.shape(single_trace.data)[0])
x_lims = [max_time-0.5,max_time+0.5]

fit_fcos =  dataset_fit_CA['data'].sel(spectral = [wv], method = "nearest").sel(time = slice(max_time-0.5,max_time+0.5))
fit_fcos = fit_fcos.values.reshape(np.shape(single_trace.data)[0])

fig, (ax, ax2) = plt.subplots(nrows=2, gridspec_kw={'height_ratios': [3, 1]}, figsize=(12, 8))
ax.scatter(time, data, color='grey', s=25, label='Coherent Artifact', alpha=0.5)
ax.plot(time, fit_fcos, color='blue', linewidth=3, label='Modeling of the Artifact')
ax.set_xlim(x_lims)


residuals = data - fit_fcos
ax2.plot(time, residuals, color='grey', label='Residual', alpha=0.5)

ax.set_xlabel("Time (ps)",fontsize=20)
ax.set_ylabel("ΔA",fontsize=20)
ax2.set_xlabel("Time (ps)",fontsize=20)
ax2.set_ylabel("Residual",fontsize=20)  
ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax2.set_xlim(ax.get_xlim()) 
ax2.set_ylim(ax.get_ylim()) 
ax.tick_params(axis='both', which='major', labelsize=15)
ax2.tick_params(axis='both', which='major', labelsize=15)


handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax.legend(handles,labels, loc='upper right',fontsize = 20)
ax2.legend(handles2, labels2, loc='upper right',fontsize = 20)
fig.suptitle(f"Modeling Coherent Artifact, solvent scan, {wv} nm",fontsize=25)
plt.tight_layout()



### Get the mean and standard error of the Solvent fit 

In [ ]:
################################################################################
####### Get the visual indication for the current fit parameters ########
################################################################################

# Step 1: Collect parameter values
param_values = {}

for params in fit_CA_param.values():
    if isinstance(params, lmfit.Parameters):
        for name, param in params.items():
            param_values.setdefault(name, []).append(param.value)

# Step 2: Compute summary statistics
summary_stats = {}
for name, vals in param_values.items():
    summary_stats[name] = {
        'mean': np.mean(vals),
        'std': np.std(vals),
        'min': np.min(vals),
        'max': np.max(vals)
    }

# Step 3: Print the statistics
print("Parameter Summary:")
for name, stats in summary_stats.items():
    print(f"{name}: mean = {stats['mean']:.6f}, std = {stats['std']:.6f}, "
          f"min = {stats['min']:.6f}, max = {stats['max']:.6f}")

# Step 4: Prepare for plotting
param_names = list(summary_stats.keys())
means = [summary_stats[name]['mean'] for name in param_names]
stds = [summary_stats[name]['std'] for name in param_names]
mins = [summary_stats[name]['min'] for name in param_names]
maxs = [summary_stats[name]['max'] for name in param_names]

# Step 5: Plot error bars + extreme values
plt.figure(figsize=(10, 6))
plt.errorbar(param_names, means, yerr=stds, fmt='o', capsize=5, markersize=6, linestyle='None', label='Mean ± Std')
plt.ylabel('Parameter Value')
plt.title('Fit Parameters: Mean ± Std with Min/Max Extremes')
plt.xticks(rotation=45)
plt.grid(True)

for i, name in enumerate(param_names):
    # Plot min and max
    plt.plot(i, mins[i], 'v', color='red', label='Min' if i == 0 else "")
    plt.plot(i, maxs[i], '^', color='green', label='Max' if i == 0 else "")
    # Annotate values
    plt.text(i + 0.1, mins[i], f"{mins[i]:.1f}", va='center', fontsize=8, color='red')
    plt.text(i + 0.1, maxs[i], f"{maxs[i]:.1f}", va='center', fontsize=8, color='green')

plt.legend()
plt.tight_layout()
plt.show();


In [ ]:
################################################################################
####### Get the visual indication for scan 1 and 2 ########
################################################################################

datasets = {
    "Scan 1 ": {
        "B": {"mean": 53.392974, "std": 24.752571, "min": 26.399270, "max": 118.742370},
        "Phi": {"mean": 155.183243, "std": 0.054658, "min": 155.058650, "max":155.314525},
        "A0": {"mean": -0.054862, "std": 0.018784, "min":  -0.128335, "max":-0.023756},
        "A1": {"mean":-0.000215, "std": 0.000737, "min":-0.002138, "max": 0.000863},
        "CA_FWHM": {"mean": 0.207204, "std":0.044764, "min": 0.128749, "max":0.303599},
        "CA_center": {"mean":0.832368, "std": 0.462718, "min":0.135989, "max":1.667889},
    },
    "Scan 2": {
        "B": {"mean": 54.535506, "std": 25.533664, "min": 24.178110, "max": 118.435084},
        "Phi": {"mean": 155.143492, "std":  0.025391, "min": 155.076266, "max": 155.185798},
        "A0": {"mean": -0.039881, "std": 0.012410, "min":  -0.067886, "max":-0.022698},
        "A1": {"mean":-0.000148, "std": 0.000752, "min":-0.002203, "max": 0.000593 },
        "CA_FWHM": {"mean": 0.226990, "std": 0.047159, "min":0.143829, "max": 0.321490},
        "CA_center": {"mean":  0.871889, "std":0.466884, "min": 0.168593, "max": 1.716871},
    },
}
# Plotting
param_names = list(next(iter(datasets.values())).keys())
dataset_names = list(datasets.keys())

fig, axs = plt.subplots(len(param_names), 1, figsize=(5, 1.8 * len(param_names)), sharex=True)


for idx, param in enumerate(param_names):
    means = [datasets[ds][param]["mean"] for ds in dataset_names]
    stds = [datasets[ds][param]["std"] for ds in dataset_names]
    mins = [datasets[ds][param]["min"] for ds in dataset_names]
    maxs = [datasets[ds][param]["max"] for ds in dataset_names]

    axs[idx].errorbar(dataset_names, means, yerr=stds, fmt='o', capsize=5, label=param)
    axs[idx].plot(dataset_names, mins, 'v', color='red', label='Min' if idx == 0 else "")
    axs[idx].plot(dataset_names, maxs, '^', color='green', label='Max' if idx == 0 else "")

    for i, (mn, mx) in enumerate(zip(mins, maxs)):
        axs[idx].text(i + 0.1, mn, f"{mn:.3f}", color='red', fontsize=8, va='center')
        axs[idx].text(i + 0.1, mx, f"{mx:.3f}", color='green', fontsize=8, va='center')

    axs[idx].set_ylabel(param)
    axs[idx].grid(True)
    axs[idx].legend(loc='center', fontsize=8)

plt.xticks(rotation=45)
plt.tight_layout(pad=1.0)
plt.show();

In [ ]:
################################################################################
####### Get the visual indication for scan 2 over time ########
################################################################################
datasets = {
    "Scan 2 - 2h30": {
        "B": {"mean": 55.079356, "std":  26.286040, "min": 24.410652, "max": 119.040987},
        "Phi": {"mean": 155.144895, "std": 0.032127, "min": 155.074176, "max": 155.215737},
        "A0": {"mean":  -0.040885, "std":  0.012417, "min":  -0.071632, "max": -0.023655},
        "A1": {"mean": -0.000137, "std":  0.000760, "min": -0.002230, "max": 0.000625},
        "CA_FWHM": {"mean": 0.225072, "std":  0.046500, "min":0.141718, "max": 0.320232},
        "CA_center": {"mean": 0.869431, "std":  0.466720, "min":  0.166413, "max":1.715461},
    },
    "Scan 2 - 2h30-5h scans": {
        "B": {"mean":  55.263603, "std":27.115866, "min": 22.948675, "max": 123.422279},
        "Phi": {"mean":  155.146802, "std": 0.031536, "min": 155.063999, "max": 155.214049},
        "A0": {"mean": -0.040609, "std": 0.011732, "min":-0.067127, "max": -0.026021},
        "A1": {"mean": -0.000147, "std":0.000813, "min":  -0.002289, "max": 0.000604},
        "CA_FWHM": {"mean": 0.224681, "std":  0.044535, "min": 0.143137, "max":0.312219},
        "CA_center": {"mean": 0.869225, "std": 0.466670, "min": 0.166151, "max": 1.712517},
    },
    "Scan 2 - 5h-7h30 scans": {
        "B": {"mean": 55.043021, "std": 25.510856, "min": 23.562987, "max": 118.306896},
        "Phi": {"mean": 155.137662, "std":  0.024624, "min": 155.067594, "max": 155.189929},
        "A0": {"mean":  -0.039612, "std": 0.012505, "min": -0.070736, "max": -0.021519},
        "A1": {"mean": -0.000142, "std":  0.000751, "min": -0.002238, "max": 0.000616},
        "CA_FWHM": {"mean": 0.227265, "std":  0.047427, "min": 0.142828, "max": 0.325149},
        "CA_center": {"mean": 0.868686, "std":  0.466896, "min": 0.165526, "max": 1.713950},
    },
    "Scan 2 - 7h30-10h scans": {
        "B": {"mean": 53.728885, "std": 25.007062, "min": 24.017510, "max": 116.617354},
        "Phi": {"mean": 155.150514, "std": 0.025877, "min": 155.077292, "max": 155.201231},
        "A0": {"mean": -0.040385, "std": 0.012742, "min": -0.069494, "max":  -0.021475 },
        "A1": {"mean": -0.000148, "std": 0.000743, "min": -0.002183, "max": 0.000599},
        "CA_FWHM": {"mean": 0.227725, "std": 0.047864, "min": 0.144425, "max":0.324326},
        "CA_center": {"mean": 0.871733, "std": 0.466551, "min": 0.169023, "max": 1.716205},
    },
        "Scan 2 - 10-12h30 scans": {
        "B": {"mean": 55.513983, "std": 25.183492, "min": 25.065456, "max": 122.397110},
        "Phi": {"mean": 155.125222, "std": 0.024078, "min": 155.078244, "max": 155.170801},
        "A0": {"mean": -0.037698, "std": 0.012471, "min": -0.065163, "max": -0.020316 },
        "A1": {"mean": -0.000163, "std":  0.000711, "min": -0.002102, "max":  0.000568},
        "CA_FWHM": {"mean": 0.231175, "std": 0.049292, "min": 0.147483, "max": 0.330273},
        "CA_center": {"mean": 0.875997, "std": 0.467353, "min": 0.172237, "max": 1.721645},
    }
}

# Plotting
param_names = list(next(iter(datasets.values())).keys())
dataset_names = list(datasets.keys())

fig, axs = plt.subplots(len(param_names), 1, figsize=(10, 2.5 * len(param_names)), sharex=True)

for idx, param in enumerate(param_names):
    means = [datasets[ds][param]["mean"] for ds in dataset_names]
    stds = [datasets[ds][param]["std"] for ds in dataset_names]
    mins = [datasets[ds][param]["min"] for ds in dataset_names]
    maxs = [datasets[ds][param]["max"] for ds in dataset_names]

    axs[idx].errorbar(dataset_names, means, yerr=stds, fmt='o', capsize=5, label=param)
    axs[idx].plot(dataset_names, mins, 'v', color='red', label='Min' if idx == 0 else "")
    axs[idx].plot(dataset_names, maxs, '^', color='green', label='Max' if idx == 0 else "")

    for i, (mn, mx) in enumerate(zip(mins, maxs)):
        axs[idx].text(i + 0.1, mn, f"{mn:.3f}", color='red', fontsize=8, va='center')
        axs[idx].text(i + 0.1, mx, f"{mx:.3f}", color='green', fontsize=8, va='center')

    axs[idx].set_ylabel(param)
    axs[idx].grid(True)
    axs[idx].legend(loc='upper right', fontsize=8)

plt.xticks(rotation=45)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show();


# Variable definition

In [ ]:
### Using Analytical solution of the convolution of IRF with exp terms for parallel model ########



def fit_4exp(params, time):
        # Extract parameters
        #exp_terms
        
        # Compute constants
        A_S2 = params['A_S2'] 
        A_S1 = params['A_S1'] 
        A_S1sq = params['A_S1sq'] 
        A_T = params['A_T'] 
        tau_S2 = params['tau_S2']
        tau_S1 = params['tau_S1']
        tau_S1sq = params['tau_S1sq']
        tau_T = params['tau_T']
        #IRF terms
        #c = params['c']
        IRF_FWHM = params['IRF_FWHM']
        t_zero = params['IRF_center']

        b = (4*np.log(2))/IRF_FWHM
        k0 = 1/tau_S2 if tau_S2 != 0 else 0
        k1 = 1/tau_S1 if tau_S1 != 0 else 0
        k2 = 1/tau_S1sq if tau_S1sq != 0 else 0
        k3 = 1/tau_T if tau_T != 0 else 0

        #H = lambda x: np.heaviside(x, 1)
        # Compute individual terms
        val  = A_S2 * np.exp((k0**2 - 4*b*k0*(time - t_zero)) / (4*b)) * norm.cdf((2*b*(time - t_zero) - k0) / np.sqrt(2*b))#* H(time - t_zero)
        val += A_S1 * np.exp((k1**2 - 4*b*k1*(time - t_zero)) / (4*b)) * norm.cdf((2*b*(time - t_zero) - k1) / np.sqrt(2*b))#* H(time - t_zero)
        val += A_S1sq * np.exp((k2**2 - 4*b*k2*(time - t_zero)) / (4*b)) * norm.cdf((2*b*(time - t_zero) - k2) / np.sqrt(2*b))#* H(time - t_zero)
        val += A_T * np.exp((k3**2 - 4*b*k3*(time - t_zero)) / (4*b)) * norm.cdf((2*b*(time - t_zero) - k3) / np.sqrt(2*b))#* H(time - t_zero)
        return val

def fit_3exp(params, time):
        # Extract parameters
        #exp_terms

        # Compute constants
        A_1 = params['A_1'] 
        A_2 = params['A_2'] 
        A_3 = params['A_3']
        tau_1 = params['tau_1']
        tau_2 = params['tau_2']
        tau_3 = params['tau_3']
        #IRF terms
        IRF_FWHM = params['IRF_FWHM']
        t_zero = params['IRF_center']

        b = (4*np.log(2))/IRF_FWHM
        k1 = 1/tau_1 if tau_1 != 0 else 0
        k2 = 1/tau_2 if tau_2 != 0 else 0
        k3 = 1/tau_3 if tau_3 != 0 else 0

        # Compute individual terms
        val = A_1 * np.exp((k1**2 - 4*b*k1*(time - t_zero)) / (4*b)) * norm.cdf((2*b*(time - t_zero) - k1) / np.sqrt(2*b))
        val += A_2 * np.exp((k2**2 - 4*b*k2*(time - t_zero)) / (4*b)) * norm.cdf((2*b*(time - t_zero) - k2) / np.sqrt(2*b))
        val += A_3 * np.exp((k3**2 - 4*b*k2*(time - t_zero)) / (4*b)) * norm.cdf((2*b*(time - t_zero) - k3) / np.sqrt(2*b))

        return val


def SK_model_4exp(params,time):
        model_vals = np.zeros((len(time)))
        decay = fit_4exp(params,time)
        XMS = Fcos(params, time)

        
        model_vals = decay + XMS

        return model_vals

def SK_model_3exp(params,time):
        model_vals = np.zeros((len(time)))
        decay = fit_3exp(params,time)
        XMS = Fcos(params, time)

        
        model_vals = decay + XMS

        return model_vals


def SK_residuals_4exp(params,time,data):
        model_vals = SK_model_4exp(params,time)
        resid = np.sqrt((model_vals - data)**2)
        return resid


def SK_residuals_3exp(params,time,data):
        model_vals = fit_3exp(params,time)
        resid = np.sqrt((model_vals - data)**2)
        return resid

# Import the Cu scan

In [ ]:
time_file = askopenfilename(filetypes=[("Text files", "*.txt")], title="Select Time vector data")
time = np.loadtxt(time_file)
time = time / 1000  # convert from fs to ps

# Get TA scan files
ta_scan_files = askopenfilenames(filetypes=[("Text files", "*.txt")], title="Select solvent scan file(s)")

if len(ta_scan_files) > 1:
    Full_Data = np.zeros((2048, len(time), len(ta_scan_files))) #Array that will contain all the 
    for n, file in enumerate(ta_scan_files):
        data = np.loadtxt(file)
        Full_Data[:, :, n] = data  # save data array to 3D array (lambda, time, scan)
    
    scan = np.mean(Full_Data, axis=2)
    scan = scan- np.mean(scan[:,1:5],axis=1)[:, np.newaxis] #Baseline correction 
else:
    scan = np.loadtxt(ta_scan_files[0])
# Now 'scan' contains the processed data

#Create an xarray to manipulate the data (much easier)
dataset = xr.Dataset(
    {
        "data": (["time","spectral",], np.transpose(scan))
    },
    coords={
        "time": time,
        "spectral": lambda_values
    }
)

# Print the dataset
print(dataset)

heatmap_interactive(dataset.time, dataset.spectral, dataset['data'].transpose('spectral','time'),'Averaged scan plot',_symlog=False)

## Plot visualization 

### Spectral dimention

In [ ]:
plot_data = dataset.data.sel(time=[1,1.3,2.2,3.6,8,10], method="nearest").sel(spectral=slice(430, 560))
ax = plot_data.plot.line(x="spectral", aspect=2, size=5)
plt.title('Cu(dchtmp)')
#plt.legend('')
plt.grid(True)
plt.ylabel('OD')
plt.show()

### Time dimension

In [ ]:
wv = 470



single_trace = dataset.sel(spectral = [wv], method="nearest") 
time = single_trace.time.values
data = single_trace.data

plt.figure()
plt.plot(time,data)
plt.xlabel('Tme (ps)')
plt.grid('On')
plt.ylabel("ΔOD")
plt.show()

# Single Kinetic trace

In [ ]:
scan_region= dataset.data.sel(spectral=slice(lower_end, higher_end))
wv = 510



single_trace = scan_region.sel(spectral = [wv], method="nearest") 
time = single_trace.time.values
data = single_trace.values.flatten()

#the A coef does need to be scaled
params = lmfit.Parameters()



params.add('B', value= fit_CA_param[single_trace.spectral.item()]['B'].value,vary = True,min=fit_CA_param[single_trace.spectral.item()]['B'].value*0.95,max=fit_CA_param[single_trace.spectral.item()]['B'].value*1.5) #determines how quickly the fringe periodicity increases 
params.add('Phi',value= fit_CA_param[single_trace.spectral.item()]['Phi'].value,min=fit_CA_param[single_trace.spectral.item()]['Phi'].value-1,max=fit_CA_param[single_trace.spectral.item()]['Phi'].value+1,vary = True) #phase shift of the modulation 
params.add('A0', value=fit_CA_param[single_trace.spectral.item()]['A0'].value , min=fit_CA_param[single_trace.spectral.item()]['A0'].value *0.5,max=fit_CA_param[single_trace.spectral.item()]['A0'].value *1.5,vary = True) #Amplitude main Gaussian. Positve results in main peak down 
params.add('A1',value= fit_CA_param[single_trace.spectral.item()]['A1'].value,min=fit_CA_param[single_trace.spectral.item()]['A1'].value*0.5,max=fit_CA_param[single_trace.spectral.item()]['A1'].value*1.5, vary = True) # help fit of asymmetric artifacts 
params.add('CA_FWHM',value= fit_CA_param[single_trace.spectral.item()]['CA_FWHM'].value,min=fit_CA_param[single_trace.spectral.item()]['CA_FWHM'].value-0.1,max=fit_CA_param[single_trace.spectral.item()]['CA_FWHM'].value+0.1 ,vary = True) # FWHM
params.add('CA_center',value=fit_CA_param[single_trace.spectral.item()]['CA_center'].value ,min=fit_CA_param[single_trace.spectral.item()]['CA_center'].value-0.2,max= fit_CA_param[single_trace.spectral.item()]['CA_center'].value+0.2,vary = True)  #Center of the chirp  

minimizer = lmfit.Minimizer(residuals_Fcos, params, fcn_args=(time, data))
result_initial = minimizer.minimize('differential_evolution')



x_lims = [0,1]

fig, (ax, ax2) = plt.subplots(nrows=2, gridspec_kw={'height_ratios': [3, 1]}, figsize=(12, 8))
ax.scatter(time, data, color='grey', s=10, label='Data', alpha=0.5)
ax.plot(time, Fcos(result_initial.params, time), color='blue', linewidth=1.5, label='Fit')
#ax.axvline(x= result.params['IRF_center'].value,color = 'k')
ax.set_xlim(x_lims)


residuals = data - Fcos(result_initial.params, time)
ax2.plot(time, residuals, color='grey', label='Residual', alpha=0.5)

ax.set_xlabel("Time (ps)")
ax.set_ylabel("ΔOD")
ax2.set_xlabel("Time (ps)")
ax2.set_ylabel("Residual")  
ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax2.set_xlim(ax.get_xlim())  

handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
all_handles = handles + handles2
all_labels = labels + labels2
by_label = dict(zip(all_labels, all_handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper right')
ax.legend(by_label.values(), by_label.keys(), loc='upper right')
fig.suptitle(f"Kinetic trace Cu(dmp)2 @ {wv}nm")

plt.tight_layout()

data_noXPM = data - Fcos(result_initial.params,time)


plt.figure(figsize=(8, 5))
plt.plot(time, data_noXPM, '-', label='Data')
plt.xlabel('Time')#plt.ylim ([-0.008,0.008])
plt.ylabel('Signal')
#plt.yscale('log')
plt.legend()
plt.show()

print(lmfit.fit_report(result_initial))

# Finetuning 

In [ ]:
scan_region= (dataset.data.sel(spectral=slice(lower_end, higher_end)))



single_trace = scan_region.sel(spectral = [wv], method="nearest") 
time = single_trace.time.values
data = single_trace.values.flatten()

# Add born for better convergance



params = lmfit.Parameters()
params.add('A_S2', value=       -0.03018143 ,min=-0.05, max=0.05,vary = False) 
params.add('tau_S2', value=        0.03990212    ,min=0.0001,max=0.3,vary = True)
params.add('A_S1', value=       0.00502558     ,min=-0.05,max = 0.05, vary = False )  
params.add('tau_S1', value=       0.20248497  ,min=0.1, max= 3, vary = False) 
params.add('A_S1sq', value=          -0.00135194      ,min=-0.05, max=0.005, vary = False)
params.add('tau_S1sq', value  =  9.39055101  ,min=3, max=15, vary = False) 
params.add('A_T', value=     0.00337824    ,max=0.006,min=0 ,vary = False) 
params.add('tau_T', value= 100,min=0,vary = False)
#params.add('c', value=0 , max = -0.00435652, vary = False )

params.add('IRF_FWHM', value= result_initial.params['CA_FWHM'].value, vary = False) 
#params.add('IRF_FWHM', value=  0.18669200, vary = False) 
params.add('IRF_center',
            value= result_initial.params['CA_center'].value, 
            #value=  0.40612069,
            min =result_initial.params['CA_center'].value,
            max= result_initial.params['CA_center'].value+0.5,
            vary = False)


params.add('B', 
           value=result_initial.params['B'].value,
           #value= 93.1645332 ,
           min=result_initial.params['B'].value* 0.8,
           max=result_initial.params['B'].value * 1.2,
           vary=False) #determines how quickly the fringe periodicity increases 

params.add('Phi', 
           value=result_initial.params['Phi'].value,
           #value= 93.1645332 ,
           min=result_initial.params['Phi'].value- 1,
           max=result_initial.params['Phi'].value+ 1,
           vary=False) #phase shift of the modulation 
 

params.add('A0', 
           value=result_initial.params['A0'].value,
           #value=-0.0774897,
           min=result_initial.params['A0'].value * 0.9,
           max=result_initial.params['A0'].value * 1.1,
           vary=False) #Amplitude main Gaussian. Positve results in main peak down 


params.add('A1', 
           value=result_initial.params['A1'].value,
           #value=4.7775e-04,
           min=result_initial.params['A1'].value* 0.9,
           max=result_initial.params['A1'].value * 1.1,
           vary=False) # help fit of asymmetric artifacts 

params.add('CA_FWHM', 
           value= result_initial.params['CA_FWHM'].value,
           min=result_initial.params['CA_FWHM'].value-0.1,
           max=result_initial.params['CA_FWHM'].value+0.1,
           vary=False)


params.add('CA_center', 
           value=result_initial.params['CA_center'].value,
           min=result_initial.params['CA_center'].value-0.1,
           max=result_initial.params['CA_center'].value+0.1,
           vary=False)  #Center of the chirp  
 


minimizer = lmfit.Minimizer(SK_residuals, params, fcn_args=(time, data))
#result = minimizer.minimize('differential_evolution')
result = minimizer.minimize()


 



print(lmfit.fit_report(result))

In [ ]:
scan_region= (dataset.data.sel(spectral=slice(lower_end, higher_end)))



single_trace = scan_region.sel(spectral = [wv], method="nearest") 
time = single_trace.time.values
data = single_trace.values.flatten()

# Add born for better convergance



params = lmfit.Parameters()
params.add('A_1', value=       0 ,min=-0.05, max=0.05,vary = False) 
params.add('tau_1', value=        0.03990212    ,min=0.0001,max=0.3,vary = False)
params.add('A_2', value=        -0.00310426    ,min=-0.05,max = 0, vary = True )  
params.add('tau_2', value=        27.4553049    ,min=1, max= 30, vary = True) 
params.add('A_3', value=         0  ,min=-0.1, max= 0, vary = False) 
params.add('tau_3', value=        100   , vary = False)




params.add('IRF_FWHM', value= result_initial.params['CA_FWHM'].value, vary = False) 
#params.add('IRF_FWHM', value=  0.18669200, vary = False) 
params.add('IRF_center',
            value= result_initial.params['CA_center'].value, 
            #value=  0.40612069,
            min =result_initial.params['CA_center'].value,
            max= result_initial.params['CA_center'].value+0.5,
            vary = False)


params.add('B', 
           value=result_initial.params['B'].value,
           #value= 93.1645332 ,
           min=result_initial.params['B'].value* 0.8,
           max=result_initial.params['B'].value * 1.2,
           vary=False) #determines how quickly the fringe periodicity increases 

params.add('Phi', 
           value=result_initial.params['Phi'].value,
           #value= 93.1645332 ,
           min=result_initial.params['Phi'].value- 1,
           max=result_initial.params['Phi'].value+ 1,
           vary=False) #phase shift of the modulation 
 

params.add('A0', 
           value=result_initial.params['A0'].value,
           #value=-0.0774897,
           min=result_initial.params['A0'].value * 0.9,
           max=result_initial.params['A0'].value * 1.1,
           vary=False) #Amplitude main Gaussian. Positve results in main peak down 


params.add('A1', 
           value=result_initial.params['A1'].value,
           #value=4.7775e-04,
           min=result_initial.params['A1'].value* 0.9,
           max=result_initial.params['A1'].value * 1.1,
           vary=False) # help fit of asymmetric artifacts 

params.add('CA_FWHM', 
           value= result_initial.params['CA_FWHM'].value,
           min=result_initial.params['CA_FWHM'].value-0.1,
           max=result_initial.params['CA_FWHM'].value+0.1,
           vary=False)


params.add('CA_center', 
           value=result_initial.params['CA_center'].value,
           min=result_initial.params['CA_center'].value-0.1,
           max=result_initial.params['CA_center'].value+0.1,
           vary=False)  #Center of the chirp  
 


minimizer = lmfit.Minimizer(SK_residuals_f2, params, fcn_args=(time, data))
result = minimizer.minimize('differential_evolution')
#result = minimizer.minimize()


 



print(lmfit.fit_report(result))

# Plots

## 4 exp terms

In [ ]:
save_figures = False
path_save_figure = '/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu/Data/Results/Cu(dchtmp)2'

In [ ]:
###############################################################################################
#####################   Full Model fit + Residuals ############################################ 
###############################################################################################

x_lims = [0.2,2]
single_trace = scan_region.sel(spectral = [wv], method="nearest") 
time = single_trace.time.values
data = single_trace.values.flatten()

fig, (ax, ax2) = plt.subplots(nrows=2, gridspec_kw={'height_ratios': [3, 1]}, figsize=(12, 8))
ax.scatter(time, data, color='grey', s=10, label='Data', alpha=0.5)
ax.plot(time, SK_model(result.params, time), color='blue', linewidth=2, label='Fit')
ax.axvline(x= result.params['IRF_center'].value,color = 'k', linewidth=2)
ax.set_xlim(x_lims)


residuals = data - SK_model(result.params, time)
ax2.plot(time, residuals, color='grey', label='Residual', alpha=0.5)

ax.set_xlabel("Time (ps)",fontsize=20)
ax.set_ylabel("ΔOD",fontsize=20)
ax2.set_xlabel("Time (ps)",fontsize=20)
ax2.set_ylabel("Residual",fontsize=20)  
ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax2.set_xlim(ax.get_xlim())  

ax.tick_params(axis='both', which='major', labelsize=20)
ax2.tick_params(axis='both', which='major', labelsize=20)

handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
all_handles = handles + handles2
all_labels = labels + labels2
by_label = dict(zip(all_labels, all_handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper right',fontsize=20)
fig.suptitle(f"Kinetic trace @ {wv}nm")
fig.suptitle(f"Fitted kinetic trace of Cu(dipp)2 at 510nm",fontsize=25)
plt.tight_layout()

if path_save_figure == True:
    plt.savefig(path_save_figure + f'/Full_{wv}.png')

In [ ]:
###############################################################################################
#####################   Model + XPM + Residuals ###############################################
############################################################################################### 

single_trace = scan_region.sel(spectral = [wv], method="nearest") 
time = single_trace.time.values
data = single_trace.values.flatten()
CA_center_val = result.params['CA_center'].value
time_corrected = time - CA_center_val

p = result.params


exp_component = f(p,time)

XMR =  Fcos(p, time)


x_lims = [-1,2]


fig, (ax, ax2) = plt.subplots(nrows=2, gridspec_kw={'height_ratios': [3, 1]}, figsize=(12, 8))
ax.scatter(time_corrected, data, color='grey', s=25, label='Data', alpha=0.5)
ax.plot(time_corrected, exp_component, color='Red', linewidth=3, label='Exp terms convoluted with IRF')
ax.plot(time, exp_component, color=[24/255,84/255,33/255], linewidth=1.5, label='Exp terms convoluted with IRF')
ax.plot(time, XMR, color=[251/255,192/255,1/255], linewidth=1.5, label='Coherent Artifact')
ax.plot(time_corrected, XMR, color='Blue', linewidth=3, label='Coherent Artifact')
ax.set_xlim(x_lims)


residuals = data - exp_component - XMR
ax2.plot(time, residuals, color='grey', label='Residual', alpha=0.5)

ax.set_xlabel("Time (ps)",fontsize=20)
ax.set_ylabel("ΔOD",fontsize=20)
ax2.set_xlabel("Time (ps)",fontsize=20)
ax2.set_ylabel("Residual",fontsize=20)  
ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax2.set_xlim(ax.get_xlim())  

ax.tick_params(axis='both', which='major', labelsize=20)
ax2.tick_params(axis='both', which='major', labelsize=20)

handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
all_handles = handles #+ handles2
all_labels = labels # + labels2
by_label = dict(zip(all_labels, all_handles))
ax.legend(by_label.values(), by_label.keys(), loc='best',fontsize=15)
fig.suptitle(f"Kinetic trace @ {wv}nm")
fig.suptitle(f"Fitted kinetic trace at 510nm",fontsize=25)
plt.tight_layout()


if path_save_figure == True:
    plt.savefig(path_save_figure + f'/componant_{wv}.png')

In [ ]:
###############################################################################################
#####################   Model + Residuals (No XPM) ############################################
############################################################################################### 


single_trace = scan_region.sel(spectral = [wv], method="nearest") 
time = single_trace.time.values
data = single_trace.values.flatten()
CA_center_val = result.params['CA_center'].value
time_corrected = time - CA_center_val

p = result.params


exp_component = f(p,time)

XMR =  Fcos(p, time)


data -= XMR

x_lims = [-1,2]


fig, (ax, ax2) = plt.subplots(nrows=2, gridspec_kw={'height_ratios': [3, 1]}, figsize=(12, 8))
ax.scatter(time_corrected, data, color='grey', s=25, label='Data with CA removed', alpha=0.5)
ax.plot(time, exp_component,color=[24/255,84/255,33/255], linewidth=2, label='Exp terms convoluted with IRF')
ax.plot(time_corrected, exp_component,color='red', linewidth=3, label='Exp terms convoluted with IRF')
ax.set_xlim(x_lims)


residuals = data - exp_component
ax2.plot(time, residuals, color='grey', label='Residual', alpha=0.5)

ax.set_xlabel("Time (ps)",fontsize=20)
ax.set_ylabel("ΔOD",fontsize=20)
ax2.set_xlabel("Time (ps)",fontsize=20)
ax2.set_ylabel("Residual",fontsize=20)  
ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax2.set_xlim(ax.get_xlim())  

ax.tick_params(axis='both', which='major', labelsize=20)
ax2.tick_params(axis='both', which='major', labelsize=20)


handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
all_handles = handles #+ #handles2
all_labels = labels #+ labels2
by_label = dict(zip(all_labels, all_handles))
ax.legend(by_label.values(), by_label.keys(), loc='best',fontsize=15)
fig.suptitle(f"Kinetic trace @ {wv}nm")
fig.suptitle(f"exp Kinetic trace without Coherent Artifact @ {wv}nm")
plt.tight_layout()

if path_save_figure == True:
    plt.savefig(path_save_figure + f'/Exp_{wv}.png')

##  3 exp terms

In [ ]:
###############################################################################################
#####################   3 exp Model fit + Residuals ############################################ 

x_lims = [0,2]
single_trace = scan_region.sel(spectral = [wv], method="nearest") 
time = single_trace.time.values
data = single_trace.values.flatten()

fig, (ax, ax2) = plt.subplots(nrows=2, gridspec_kw={'height_ratios': [3, 1]}, figsize=(12, 8))
ax.scatter(time, data, color='grey', s=10, label='Data', alpha=0.5)
ax.plot(time, SK_model_f2(result.params, time), color='blue', linewidth=2, label='Fit')
#ax.axvline(x= result.params['IRF_center'].value,color = 'k')
ax.set_xlim(x_lims)


residuals = data - SK_model_f2(result.params, time)
ax2.plot(time, residuals, color='grey', label='Residual', alpha=0.5,linewidth = 2)

ax.set_xlabel("Time (ps)",fontsize=20)
ax.set_ylabel("ΔOD",fontsize=20)
ax2.set_xlabel("Time (ps)",fontsize=20)
ax2.set_ylabel("Residual",fontsize=20)  
ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax2.set_xlim(ax.get_xlim())  

ax.tick_params(axis='both', which='major', labelsize=20)
ax2.tick_params(axis='both', which='major', labelsize=20)

handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
all_handles = handles + handles2
all_labels = labels + labels2
by_label = dict(zip(all_labels, all_handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper right',fontsize=20)
#fig.suptitle(f"Kinetic trace @ {wv}nm")
fig.suptitle(f"Fitted kinetic trace of Cu(dipp)2 at 510nm",fontsize=25)
#plt.savefig(f'/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteof
##plt.savefig(f'/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu/Data/Results/Cu(dchtmp)2/Full_{wv}.png')
plt.tight_layout()


# Save & import the fits

In [ ]:
import pickle

# Save the result of the fit 
with open(f'lmfit_result_Cu(dchtmp)_{wv}_original', 'wb') as f:
    pickle.dump(result, f)

In [ ]:
import pickle

with open(r'/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu/Data/Results/Fit_results/lmfit_result_Cu(dipp)_510_original', 'rb') as file:
    # Load the data from the file
    result = pickle.load(file)

# Global Fit - To be tried later

In [ ]:



def global_model(params_exp,params_CA,params_IRF_center, param_IRF_FWHM,time,wavelengths):

    tau_S2 = params_exp['tau_S2']
    
    tau_S1 = params_exp['tau_S1']
    
    tau_S1sq = params_exp['tau_S1sq']
    
    tau_T = params_exp['tau_T']
    c = params_exp['c']

    model_vals = np.zeros((len(t), len(wavelengths)))
    for w_idx, lam in enumerate(wavelengths):
        params_CA_wv = params_CA[lam]
        XMS = Fcos(params_CA_wv, time)

        model_vals[:, w_idx] += XMS

        IRF_FWHM = lin(lam,param_IRF_FWHM)
        IRF_Center = poly4(lam,params_IRF_center)
        IRF = irf(time,IRF_FWHM, IRF_Center)
        
        A_S2 = params_exp['A_S2_{i}'] for i in wavelengths
        A_S1 = params_exp['A_S1_{i}'] for i in wavelengths
        A_S1sq = params_exp['A_S1sq_{i}'] for i in wavelengths
        A_T = params_exp['A_T_{i}'] for i in wavelengths

        decay = exp_terms(time,A_S2,tau_S2,A_S1,tau_S1,A_S1sq,tau_S1sq,A_T,tau_T,c)
        model_vals += np.convolve(IRF, decay, mode='same')

    return model_vals.ravel()


def residuals(params_exp,params_CA,params_IRF_center, param_IRF_FWHM,time,wavelengths,data):
    model_vals = global_model(params_exp,params_CA,params_IRF_center, param_IRF_FWHM,time,wavelengths)
    resid = np.sqrt((model_vals - data.ravel())**2)
    return resid.revel()


In [ ]:
data = dataset.data.sel(spectral=slice(469, 560))
time  = data.time.values
wavelengths = data.spectral.values


# Plot of final results

In [ ]:

# Example data
experiment_names = ['Cu(dmp)2', 'Cu(dpp)2', 'Cu(dipp)2']
means = [55, 132, 57]
standard_deviations = [11, 43, 19]

# Define a list of RGB colors for each experiment name
rgb_colors = [(0.0,0.8,0.588), (1, 0.631, 0.353), (0.937,0.334,0.231)]  # Example RGB colors

# Create a plot with error bars
plt.figure(figsize=(10, 6))

# Plot each experiment with its corresponding color
for i, name in enumerate(experiment_names):
    plt.errorbar(name, means[i], yerr=standard_deviations[i], fmt='o', capsize=8, color=rgb_colors[i], ecolor=rgb_colors[i], elinewidth=4, markersize=8)

# Add titles and labels
plt.ylabel('S2 lifetime in fs', fontsize=20)

# Adjust x-axis limits to remove extra space
plt.xlim(-0.5, len(experiment_names) - 0.5)

# Change the tick size and x-label size
plt.tick_params(axis='x', labelsize=20)  # Change x-tick label size
plt.tick_params(axis='y', labelsize=20)  # Change y-tick label size (optional)
plt.xlabel('', fontsize=20)  # Change x-label size

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()


